In [ ]:
# Install alpaca-py if needed
# !pip install alpaca-py

from alpaca.data import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

## 1. Set Up Alpaca Connection

Get your API keys from: https://app.alpaca.markets/paper/dashboard/overview

In [ ]:
# YOUR API KEYS (Paper trading)
API_KEY = "YOUR_API_KEY_HERE"
API_SECRET = "YOUR_SECRET_KEY_HERE"

# Create client
client = StockHistoricalDataClient(API_KEY, API_SECRET)

print("✅ Alpaca client created")

## 2. Test Data Download for YOUR Positions

In [ ]:
# Your current positions
tickers = ['MU', 'ASTS', 'LUNR', 'SPY', 'QQQ']

# Request parameters
request_params = StockBarsRequest(
    symbol_or_symbols=tickers,
    timeframe=TimeFrame.Day,
    start=datetime.now() - timedelta(days=100),
    end=datetime.now()
)

# Download data
bars = client.get_stock_bars(request_params)
df = bars.df

print(f"Downloaded {len(df)} bars for {len(tickers)} tickers")
df.head()

## 3. Calculate Factors (same as our research)

In [ ]:
def calc_live_factors(ticker_df):
    """Calculate all factors for a single ticker"""
    df = ticker_df.copy()
    
    # RSI
    delta = df['close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # EMAs
    df['EMA_200'] = df['close'].ewm(span=200).mean()
    df['EMA_50'] = df['close'].ewm(span=50).mean()
    df['EMA_21'] = df['close'].ewm(span=21).mean()
    
    # Momentum
    df['Ret_20'] = df['close'].pct_change(20)
    df['Ret_60'] = df['close'].pct_change(60)
    
    # Volatility
    df['Vol'] = df['close'].pct_change().rolling(20).std() * np.sqrt(252)
    df['Vol_Rank'] = df['Vol'].rolling(252).rank(pct=True)
    df['LowVol'] = df['Vol_Rank'] < 0.3
    
    # 52-week levels
    df['High_52w'] = df['high'].rolling(252).max()
    df['Pct_From_High'] = (df['close'] - df['High_52w']) / df['High_52w']
    
    # Z-Score
    df['ZScore'] = (df['close'] - df['close'].rolling(20).mean()) / df['close'].rolling(20).std()
    
    return df

# Apply to each ticker
results = []
for ticker in tickers:
    ticker_data = df.xs(ticker, level='symbol')
    ticker_factors = calc_live_factors(ticker_data)
    ticker_factors['ticker'] = ticker
    results.append(ticker_factors)

live_df = pd.concat(results)
print("✅ Factors calculated")
live_df.tail()

## 4. Apply OUR Best Strategies to LIVE Data

In [ ]:
# Get most recent data for each ticker
current = live_df.groupby('ticker').tail(1)

print("="*70)
print("LIVE STRATEGY SIGNALS (December 19, 2025)")
print("="*70)

for ticker in tickers:
    row = current[current['ticker'] == ticker].iloc[0]
    
    print(f"\n📊 {ticker}:")
    print(f"   Price: ${row['close']:.2f}")
    print(f"   RSI: {row['RSI']:.1f}")
    print(f"   Z-Score: {row['ZScore']:.2f}")
    print(f"   From 52wk High: {row['Pct_From_High']*100:.1f}%")
    print(f"   Above EMA200: {row['close'] > row['EMA_200']}")
    
    # Check our best strategies
    signals = []
    
    # Strategy 1: AboveEMA200 (t=62.17)
    if row['close'] > row['EMA_200']:
        signals.append("✅ BULLISH: Above EMA200")
    else:
        signals.append("⚠️ BEARISH: Below EMA200")
    
    # Strategy 2: LowVol (t=61.44)
    if row['LowVol']:
        signals.append("✅ BULLISH: Low Volatility")
    
    # Strategy 3: Near 52wk High (t=82.89)
    if row['Pct_From_High'] > -0.05:
        signals.append("✅ BULLISH: Near 52-week high")
    
    # Strategy 4: RSI Extremes (NEW - our addition)
    if row['RSI'] >= 80:
        signals.append("🔴 OVERBOUGHT: RSI >= 80 - TAKE PROFITS")
    elif row['RSI'] >= 70:
        signals.append("⚠️ OVERBOUGHT: RSI >= 70 - Consider trimming")
    elif row['RSI'] <= 20:
        signals.append("🟢 OVERSOLD: RSI <= 20 - Bounce candidate")
    
    # Strategy 5: Multi-factor (Sharpe 2.04)
    if row['RSI'] < 30 and row['LowVol']:
        signals.append("🎯 MULTI-FACTOR BUY: Oversold + LowVol")
    
    print("\n   Signals:")
    for sig in signals:
        print(f"      {sig}")

## 5. Test Paper Trade Execution

In [ ]:
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

# Create trading client (PAPER ACCOUNT)
trading_client = TradingClient(API_KEY, API_SECRET, paper=True)

# Check account
account = trading_client.get_account()
print(f"✅ Paper Account Connected")
print(f"   Cash: ${float(account.cash):,.2f}")
print(f"   Portfolio Value: ${float(account.portfolio_value):,.2f}")

# Get current positions
positions = trading_client.get_all_positions()
print(f"\n   Current Positions: {len(positions)}")
for pos in positions:
    print(f"      {pos.symbol}: {pos.qty} shares @ ${float(pos.current_price):.2f}")

## 6. Example Paper Trade (DO NOT RUN until ready!)

In [ ]:
# EXAMPLE: Buy 1 share of SPY as test
# Uncomment to execute:

# order_data = MarketOrderRequest(
#     symbol="SPY",
#     qty=1,
#     side=OrderSide.BUY,
#     time_in_force=TimeInForce.DAY
# )
# 
# order = trading_client.submit_order(order_data)
# print(f"✅ Order submitted: {order.id}")

print("⚠️ Paper trade code ready but commented out")

## NEXT STEPS

1. Run this notebook to test Alpaca connection
2. Verify you can download data and calculate factors
3. Check paper account is working
4. Once regime validation completes, we'll know WHICH strategies to trade
5. Then: automated daily signal generation → paper trades